# 101 — Segmentación, metadatos y ventanas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Paso `N − S = 300`. Intervalos: `[0,400)`, `[300,700)`, `[600,1000)`,
`[900,1200)` → **4 chunks** (`⌈(1200−100)/300⌉ = 4`). Tokens almacenados:
`400·3 + 300 = 1500` → sobrecoste `1500/1200 = 1.25×`.

**Ejercicio 2.** La definición `[280, 420)` mide 140 tokens. `[0,400)` la corta en 400;
`[300,700)` empieza en 300 > 280. **Ningún chunk la contiene** con `S = 100`. Con
`S = 200` (paso 200): `[0,400)`, `[200,600)` ⊇ `[280,420)` ✓. Garantía general: una idea
de longitud `L` cabe entera en algún chunk si `L ≤ S` no basta por sí solo; la condición
segura es `L ≤ N − paso = S`... y que la idea no empiece en el primer tramo perdido. En
la práctica: **S ≥ longitud de la idea más larga que no quieres cortar**, o fronteras
estructurales.

**Ejercicio 3.** (a) Estructural: los encabezados y tablas de Markdown son fronteras
naturales y cortarlas invalida el contenido. (b) Semántica (o fijo con solape amplio):
sin puntuación fiable no hay estructura que respetar; el cambio de tema se detecta con
embeddings. (c) Parent-document: el hijo (cláusula) da la cita exacta; el padre
(sección) da el contexto para interpretarla.

**Ejercicio 4.** Verificación de contrato en el código: claves estables, valores
dependientes de la semilla.


In [ ]:
result = run_lab("retrieval", seed=101)
assert result["kind"] == "retrieval"
assert result["evidence"]
show(result)


In [ ]:
def chunks(total, n, s):
    paso = n - s
    out, inicio = [], 0
    while inicio < total:
        out.append((inicio, min(inicio + n, total)))
        if inicio + n >= total:
            break
        inicio += paso
    return out

cs = chunks(1200, 400, 100)
almacenados = sum(b - a for a, b in cs)
print("chunks:", cs)
print("n =", len(cs), " sobrecoste =", round(almacenados / 1200, 2))

def contiene(cs, ini, fin):
    return [c for c in cs if c[0] <= ini and fin <= c[1]]

print("S=100 contiene [280,420)? ->", contiene(cs, 280, 420))
print("S=200 contiene [280,420)? ->", contiene(chunks(1200, 400, 200), 280, 420))


## Reflexión

1. Si duplicas el solape `S` manteniendo `N`, ¿qué mejora, qué empeora y cómo lo medirías con consultas etiquetadas?
2. ¿Por qué sentence-window puede recuperar mejor que chunks de 512 tokens y a la vez darle al LLM más contexto que ellos? ¿Dónde está la trampa en coste?
3. ¿Qué metadatos son imprescindibles para que la clase 105 pueda citar "documento X, sección Y" sin reprocesar el corpus?
